# Notebook 2: Regression with PyTorch

In the previous notebook, we built a neural network from scratch. Now, we will transition to using **PyTorch**, a powerful deep learning framework.

**What you will learn**
- how the hand-written pieces of Notebook 1 map onto PyTorch
- how automatic differentiation (autograd) replaces our `backward` methods
- how to write a standard PyTorch training loop with mini-batches and a validation set
- how a network can predict a full probability distribution instead of a single number

**Runtime:** about 10 seconds on a laptop.

## Key Concepts in PyTorch

1.  **Tensors**: The fundamental data structure, similar to NumPy arrays but with GPU support.
2.  **Autograd**: Automatic differentiation engine that computes gradients for us.
3.  **nn.Module**: The base class for all neural network modules.
4.  **Optimizers**: Pre-implemented optimization algorithms (SGD, Adam, etc.) in `torch.optim`.
5.  **DataLoaders**: Efficient data handling tools in `torch.utils.data`.

## From Notebook 1 to PyTorch

Everything we implemented by hand has a direct counterpart:

| Notebook 1 (NumPy) | PyTorch |
|---|---|
| `Linear(1, 16)` | `nn.Linear(1, 16)` |
| `ReLU()`, `LeakyReLU()`, `Sigmoid()` | `nn.ReLU()`, `nn.LeakyReLU()`, `nn.Sigmoid()` |
| list of layers and `predict(network, X)` | `nn.Sequential(...)` and `model(X)` |
| `MSE()` | `nn.MSELoss()` |
| `backward` method of every layer | `loss.backward()` (autograd) |
| `layer.grad_weights` | `layer.weight.grad` |
| `SGD(...)`, `Adam(...)` and `optimizer.step(network)` | `optim.SGD(...)`, `optim.Adam(...)` and `optimizer.step()` |
| `l2_lambda` | `weight_decay` (equal to $2\lambda_2$) |
| full dataset in every step | mini-batches from a `DataLoader` |

## Task: Redshift to Distance Regression (Revisited)

We will solve the same regression problem as before, but using PyTorch's high-level building blocks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# Set seed
torch.manual_seed(42)
np.random.seed(42)

# Our model is tiny, so it does not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## 1. Autograd in a Few Lines

In Notebook 1 we derived and implemented a `backward` method for every layer. PyTorch does this automatically: every operation on a tensor with `requires_grad=True` is recorded, and calling `.backward()` on the result runs backpropagation through all recorded operations. The gradients end up in the `.grad` attribute.

Below we check this for a simple function and for the linear layer of Notebook 1.

In [ ]:
# A scalar example: y = x^3 + 2x, so dy/dx = 3x^2 + 2
x = torch.tensor(2.0, requires_grad=True)    # requires_grad: track all operations on x
y = x**3 + 2 * x
y.backward()                                 # backpropagation through everything that led to y
print(f"autograd: dy/dx = {x.grad.item():.1f}, by hand: {3 * 2.0**2 + 2:.1f}")

# The linear layer from Notebook 1: Z = X W, with the loss L = mean(Z^2)
X = torch.randn(5, 3)
W = torch.randn(3, 2, requires_grad=True)
Z = X @ W
loss = (Z**2).mean()
loss.backward()

dZ = 2 * Z.detach() / Z.numel()              # dL/dZ, by hand
dW = X.T @ dZ                                # the backpropagation formula from Notebook 1
print("autograd agrees with dW = X^T dZ:", torch.allclose(W.grad, dW))

## 2. Preparing the Data

We generate the same toy data as in Notebook 1 (with more samples), split it into training and validation data, normalize it and wrap the training data in a `DataLoader`, which shuffles the data and splits it into mini-batches for us.

In [ ]:
noise_std = 500   # Mpc

def true_relation(z):
    return 3000 * z + 500 * z**2

# Generate Data (same relation as before, but more samples)
def generate_data(n_samples=1000):
    z = np.random.uniform(0, 3, n_samples)
    noise = np.random.normal(0, noise_std, n_samples)
    D = true_relation(z) + noise
    return z.reshape(-1, 1).astype(np.float32), D.reshape(-1, 1).astype(np.float32)

X_np, y_np = generate_data()

# Split into training (80%) and validation (20%) data
n_train = 800

# Normalize (using the statistics of the training data)
X_mean, X_std = X_np[:n_train].mean(), X_np[:n_train].std()
y_mean, y_std = y_np[:n_train].mean(), y_np[:n_train].std()

X_scaled = (X_np - X_mean) / X_std
y_scaled = (y_np - y_mean) / y_std

# Convert to PyTorch Tensors
X_tensor = torch.from_numpy(X_scaled)
y_tensor = torch.from_numpy(y_scaled)
X_train, y_train = X_tensor[:n_train], y_tensor[:n_train]
X_val, y_val = X_tensor[n_train:], y_tensor[n_train:]

# Create DataLoader for the training data
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Training data shape: {X_train.shape}, {y_train.shape}")
print(f"Validation data shape: {X_val.shape}, {y_val.shape}")
print(f"Number of mini-batches per epoch: {len(train_loader)}")

## 3. Defining the Model with `nn.Module`

We define a class inheriting from `nn.Module`. In the `__init__`, we define the layers. In `forward`, we define how the data flows through the layers. There is no `backward` method: autograd takes care of it.

In [ ]:
class GalaxyNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GalaxyNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

# Instantiate
model = GalaxyNet(input_dim=1, hidden_dim=32, output_dim=1)
print(model)

## 4. Training Loop

We use `nn.MSELoss` and `optim.Adam`. The training loop involves:
1.  Zeroing gradients (`optimizer.zero_grad()`): PyTorch adds up gradients from successive `backward()` calls, so we have to reset them in every step.
2.  Forward pass (`model(x)`)
3.  Loss calculation (`criterion(pred, y)`)
4.  Backward pass (`loss.backward()`)
5.  Optimizer step (`optimizer.step()`)

In Notebook 1 we used the full dataset for every update. Here, one **epoch** is one pass through the training data in mini-batches of 32 samples, i.e. 25 optimizer steps per epoch. After every epoch we evaluate the loss on the validation data.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
train_loss_history, val_loss_history = [], []

for epoch in range(epochs):
    model.train() # Set to training mode
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        # 1. Zero gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        predictions = model(batch_X)
        
        # 3. Compute loss
        loss = criterion(predictions, batch_y)
        
        # 4. Backward pass
        loss.backward()
        
        # 5. Optimizer step
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_loss_history.append(epoch_loss / len(train_loader))

    # Validation loss: no gradients needed, the validation set is small enough to use it in one go
    model.eval() # Set to evaluation mode
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val).item()
    val_loss_history.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Training Loss: {train_loss_history[-1]:.5f}, Validation Loss: {val_loss:.5f}")

plt.plot(train_loss_history, label='Training')
plt.plot(val_loss_history, label='Validation')
plt.axhline((noise_std / y_std)**2, color='black', ls=':', label='Noise level')
plt.yscale('log')
plt.title('PyTorch Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE (normalized units)')
plt.legend()
plt.show()

In [ ]:
# Evaluation on a regular grid of redshifts
z_grid = np.linspace(0, 3, 200, dtype=np.float32).reshape(-1, 1)
model.eval() # Set to evaluation mode
with torch.no_grad(): # Disable gradient calculation for efficiency
    preds_scaled = model(torch.from_numpy((z_grid - X_mean) / X_std)).numpy()

# Denormalize
preds = preds_scaled * y_std + y_mean

plt.figure(figsize=(8, 5))
plt.scatter(X_np, y_np, s=10, color='gray', alpha=0.5, label='Data')
plt.plot(z_grid, preds, color='blue', lw=2, label='PyTorch Model')
plt.plot(z_grid, true_relation(z_grid), color='black', ls='--', label='True relation')
plt.xlabel('Redshift (z)')
plt.ylabel('Distance (Mpc)')
plt.title('PyTorch Regression Results')
plt.legend()
plt.show()

## 5. Predicting Uncertainties: from Regression to Density Estimation

So far the network predicts a single number for each redshift. But the data are noisy: at a given redshift there is a whole *distribution* of possible distances, $p(D|z)$. Can a network learn this distribution, and not only its mean?

The simplest approach is to assume a Gaussian whose mean $\mu(z)$ **and** standard deviation $\sigma(z)$ are both outputs of the network. We then train the network by making the observed data as probable as possible, i.e. by minimizing the **negative log-likelihood (NLL)** of the training data:

$$ \mathcal{L} = -\frac{1}{N} \sum_i \log p(D_i|z_i) = \frac{1}{N} \sum_i \left[ \log \sigma(z_i) + \frac{(D_i - \mu(z_i))^2}{2 \sigma(z_i)^2} \right] + \text{const} $$

A few remarks:
- If $\sigma$ is a fixed constant, this loss is the MSE (up to a factor and a constant). MSE regression is therefore maximum likelihood with constant Gaussian noise.
- The network outputs $\log \sigma$ instead of $\sigma$, which guarantees a positive standard deviation.
- To make the example more interesting, we create a new dataset in which the noise increases with redshift (distant galaxies are fainter and harder to measure). The network has to find out by itself how large the noise is at each redshift.

In [ ]:
def sigma_true(z):
    # The measurement error now grows with redshift (in Mpc)
    return 150 + 300 * z

# New dataset with redshift-dependent noise
z_h = np.random.uniform(0, 3, (2000, 1)).astype(np.float32)
D_h = (true_relation(z_h) + sigma_true(z_h) * np.random.normal(size=z_h.shape)).astype(np.float32)

# We reuse the normalization from above
X_h = torch.from_numpy((z_h - X_mean) / X_std)
y_h = torch.from_numpy((D_h - y_mean) / y_std)
X_h_train, y_h_train = X_h[:1600], y_h[:1600]
X_h_val, y_h_val = X_h[1600:], y_h[1600:]
train_loader_h = DataLoader(TensorDataset(X_h_train, y_h_train), batch_size=32, shuffle=True)

class GaussianNet(nn.Module):
    """Predicts the mean and the log standard deviation of a Gaussian p(D|z)."""
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)       # two outputs: mean and log(sigma)
        )

    def forward(self, x):
        mean, log_sigma = self.net(x).chunk(2, dim=1)
        return mean, log_sigma

def gaussian_nll(mean, log_sigma, y):
    # -log N(y | mean, sigma^2), without the constant 0.5 * log(2 pi)
    return (log_sigma + 0.5 * ((y - mean) / log_sigma.exp())**2).mean()

gaussian_model = GaussianNet()
optimizer = optim.Adam(gaussian_model.parameters(), lr=0.001)

# Exactly the same training loop as before, only the loss function changed
for epoch in range(100):
    gaussian_model.train()
    for batch_X, batch_y in train_loader_h:
        optimizer.zero_grad()
        mean, log_sigma = gaussian_model(batch_X)
        loss = gaussian_nll(mean, log_sigma, batch_y)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 20 == 0:
        gaussian_model.eval()
        with torch.no_grad():
            train_nll = gaussian_nll(*gaussian_model(X_h_train), y_h_train).item()
            val_nll = gaussian_nll(*gaussian_model(X_h_val), y_h_val).item()
        print(f"Epoch {epoch+1}/100, Training NLL: {train_nll:.4f}, Validation NLL: {val_nll:.4f}")

In [ ]:
z_grid = np.linspace(0, 3, 200, dtype=np.float32).reshape(-1, 1)
gaussian_model.eval()
with torch.no_grad():
    mean, log_sigma = gaussian_model(torch.from_numpy((z_grid - X_mean) / X_std))

# Back to physical units (Mpc)
mean = mean.numpy().ravel() * y_std + y_mean
sigma = log_sigma.exp().numpy().ravel() * y_std
z_grid = z_grid.ravel()

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].scatter(z_h, D_h, s=5, color='gray', alpha=0.4, label='Data')
axs[0].plot(z_grid, mean, color='red', label=r'Predicted mean $\mu(z)$')
axs[0].fill_between(z_grid, mean - sigma, mean + sigma, color='red', alpha=0.3, label=r'$\pm 1\sigma(z)$')
axs[0].fill_between(z_grid, mean - 2 * sigma, mean + 2 * sigma, color='red', alpha=0.15, label=r'$\pm 2\sigma(z)$')
axs[0].plot(z_grid, true_relation(z_grid), color='black', ls='--', label='True relation')
axs[0].set(xlabel='Redshift (z)', ylabel='Distance (Mpc)', title=r'Predicted distribution $p(D|z)$')
axs[0].legend()

axs[1].plot(z_grid, sigma, color='red', label=r'Predicted $\sigma(z)$')
axs[1].plot(z_grid, sigma_true(z_grid), color='black', ls='--', label=r'True $\sigma(z)$')
axs[1].set(xlabel='Redshift (z)', ylabel='Standard deviation (Mpc)', ylim=(0, None), title='Predicted vs true noise level')
axs[1].legend()
plt.show()

## Summary

- PyTorch provides the building blocks we wrote by hand in Notebook 1. The most important one is autograd: we only write the forward pass, `loss.backward()` does the rest.
- The training loop always has the same five steps: zero the gradients, forward pass, loss, backward pass, optimizer step.
- Mini-batches give many (noisy) updates per epoch. The validation loss tells us how well the model generalizes.
- A network can predict a *probability distribution* instead of a single number: it outputs the parameters of the distribution (here the mean and standard deviation of a Gaussian) and is trained by minimizing the negative log-likelihood of the training data.

**Next:** the last point is the central idea of the remaining notebooks. A Gaussian is a very restrictive choice: it is symmetric and has a single peak. In Notebook 3 we build *normalizing flows*, which can represent almost arbitrary distributions and are trained with exactly the same loss. In Notebooks 4 and 5 the distribution we learn will be the posterior $p(\theta|x)$ of model parameters $\theta$ given data $x$.